In [1]:
import importlib
import ms_fl_scraper
ms_fl_scraper = importlib.reload(ms_fl_scraper)
scrape_section_url_async = ms_fl_scraper.scrape_section_url_async

import asyncio
import threading
import sys

In [2]:
def run_scraper(url, state, output_file):
    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
        asyncio.set_event_loop(loop)
    else:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    
    try:
        loop.run_until_complete(scrape_section_url_async(
            section_url=url,
            state=state,
            output_file=output_file
        ))
        return True
    except Exception as e:
        print(f"[SCRAPE FAILED] {state} {url}: {e}")
        return False
    finally:
        loop.close()

In [3]:
#t = threading.Thread(target=run_scraper)
#t.start()
#t.join()

cool. können wir jetzt ein loop schreiben? das soll über usstates50.xlsx gehen und für alle staaten die spalten cn_source, el_source el2_source und le_source bearbeiten, sofern dort ein link zu findlaw drin ist und den scraper aufrufen. alle output jsonl files sollen in staats-spezifischen unterordnern in einem neuen ordner "us_codes" landen. passt?

In [4]:
import pandas as pd
from tqdm.notebook import tqdm
import os
import shutil

In [5]:
# we want to collect all links in usstates50xlsx. cols cn_source, el_source, el2_source, le_source if they refer to findlaw pages and put them in a long format dataframe with columns state, type, url; type should be the column name where the url was found, e.g. cn_source, el_source, el2_source, le_source but without _source

us = pd.read_excel("usstates50.xlsx")
links = []
for index, row in us.iterrows():
    for col in ["cn_source", "el_source", "el2_source", "le_source"]:
        url = row[col]
        if isinstance(url, str) and "findlaw" in url:
            links.append({
                "state": row["state"],
                "type": col.rstrip("_source"),
                "url": url
            })
links_df = pd.DataFrame(links)

# create list of all unique urls

urls = links_df["url"].unique()

In [ ]:
## scrape all urls in loop
## all scraped urls are stored in directory state_codes/STATE/type.jsonl
## we always refer to the first instance of the url in the dataframe, duplicate urls are only copied later on

for url in tqdm(urls):
    state = links_df[links_df["url"] == url]["state"].iloc[0]
    type = links_df[links_df["url"] == url]["type"].iloc[0]
    # if directory does not exist create it
    if not os.path.exists(f"state_codes/{state}"):
        os.makedirs(f"state_codes/{state}")
    output_file = f"state_codes/{state}/{type}.jsonl"
    # scrape and save file
    # but only if the file does not already exist, otherwise we can skip it
    if os.path.exists(output_file):
        continue
    else:
        t = threading.Thread(target=run_scraper, args=(url, state, output_file))
        t.start()
        t.join()

    # if scraping failed (e.g., timeout/redirect), do not continue with copy logic
    if not os.path.exists(output_file):
        continue

    # check whether there is another instance
    if len(links_df[links_df["url"] == url]) > 1:
        # if there is another instance, copy the file to the other location
        for index, row in links_df[links_df["url"] == url].iterrows():
            if row["state"] != state or row["type"] != type:
                other_state = row["state"]
                other_type = row["type"]
                other_output_file = f"state_codes/{other_state}/{other_type}.jsonl"
                # if directory does not exist create it
                if not os.path.exists(f"state_codes/{other_state}"):
                    os.makedirs(f"state_codes/{other_state}")
                shutil.copyfile(output_file, other_output_file)

  0%|          | 0/123 [00:00<?, ?it/s]

Loading https://codes.findlaw.com/ks/chapter-46-legislature/ ...
Found 234 statutes. Fetching...


Fetching (Playwright x4): 100%|██████████| 234/234 [00:43<00:00,  5.43it/s]


Done. Saved to state_codes/Kansas/l.jsonl
Loading https://codes.findlaw.com/ky/kentucky-constitution/ ...
Found 247 statutes. Fetching...


Fetching (Playwright x4): 100%|██████████| 247/247 [01:15<00:00,  3.25it/s]


...

Done. Saved to state_codes/Kentucky/el.jsonl
Loading https://codes.findlaw.com/ky/title-ii-legislative-branch/ ...
Found 341 statutes. Fetching...


Fetching (Playwright x4): 100%|██████████| 341/341 [01:14<00:00,  4.56it/s]


Done. Saved to state_codes/Kentucky/l.jsonl
Loading https://codes.findlaw.com/la/louisiana-constitution-of-1974/ ...
Found 336 statutes. Fetching...


Fetching (Playwright x4): 100%|██████████| 336/336 [01:11<00:00,  4.69it/s]


Done. Saved to state_codes/Louisiana/cn.jsonl
Loading https://codes.findlaw.com/la/revised-statutes/ ...


In [5]:
# Diagnose: Kansas constitution URL(s) aus der Excel-Tabelle
ks_rows = links_df[(links_df["state"].astype(str).str.lower() == "kansas") & (links_df["type"] == "cn")]
print(ks_rows[["state", "type", "url"]].drop_duplicates().to_string(index=False))

NameError: name 'links_df' is not defined

In [6]:
# Einzeldokument-Test: Kansas Constitution
ks_url = "https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/"
ks_state = "Kansas"
ks_type = "cn"
os.makedirs(f"us_codes/{ks_state}", exist_ok=True)
ks_output = f"us_codes/{ks_state}/{ks_type}.jsonl"

if os.path.exists(ks_output):
    os.remove(ks_output)

ok = run_scraper(ks_url, ks_state, ks_output)
print("run_scraper ok:", ok)
print("file exists:", os.path.exists(ks_output))
if os.path.exists(ks_output):
    print("file size:", os.path.getsize(ks_output))

[SCRAPE FAILED] Kansas https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/: Cannot run the event loop while another loop is running
run_scraper ok: False
file exists: False


C:\Users\TimoS\AppData\Local\Temp\ipykernel_25184\1770054904.py:18: RuntimeWarning: coroutine 'scrape_section_url_async' was never awaited
  return False


In [7]:
import threading
import asyncio
from playwright.async_api import async_playwright

diag = {}

def ks_nav_diag(url):
    async def _go():
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
            context = await browser.new_context()
            page = await context.new_page()
            resp = await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            diag["status"] = None if resp is None else resp.status
            diag["final_url"] = page.url
            diag["title"] = await page.title()
            diag["has_tree_root"] = await page.locator(".fl-expandable-tree-accordion").count() > 0
            diag["has_tree_items"] = await page.locator(".fl-expandable-tree-accordion > .fl-accordion-item").count() > 0
            body_text = await page.locator("body").inner_text()
            diag["body_excerpt"] = body_text[:500]
            await context.close()
            await browser.close()

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(_go())
    finally:
        loop.close()

t = threading.Thread(target=ks_nav_diag, args=(ks_url,))
t.start()
t.join()
diag

Task exception was never retrieved
future: <Task finished name='Task-59' coro=<Connection.run() done, defined at d:\Sync\Uni\Promotion\research\lawscraper\.venv\Lib\site-packages\playwright\_impl\_connection.py:305> exception=NotImplementedError()>
Traceback (most recent call last):
  File "d:\Sync\Uni\Promotion\research\lawscraper\.venv\Lib\site-packages\playwright\_impl\_connection.py", line 312, in run
    await self._transport.connect()
  File "d:\Sync\Uni\Promotion\research\lawscraper\.venv\Lib\site-packages\playwright\_impl\_transport.py", line 133, in connect
    raise exc
  File "d:\Sync\Uni\Promotion\research\lawscraper\.venv\Lib\site-packages\playwright\_impl\_transport.py", line 120, in connect
    self._proc = await asyncio.create_subprocess_exec(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "C:\Users\TimoS\AppData\Local\Programs\Python\Python313\Lib\asyncio\subprocess.py", line 224, in create_subprocess_exec
    transport, p

{}

In [8]:
# Kansas-Test im gleichen Thread-Muster wie der Batch-Loop
ks_output = "us_codes/Kansas/cn.jsonl"
os.makedirs("us_codes/Kansas", exist_ok=True)
if os.path.exists(ks_output):
    os.remove(ks_output)

result_box = {}
def _runner():
    result_box["ok"] = run_scraper(ks_url, "Kansas", ks_output)

t = threading.Thread(target=_runner)
t.start()
t.join()

print("thread ok:", result_box.get("ok"))
print("file exists:", os.path.exists(ks_output))
if os.path.exists(ks_output):
    print("file size:", os.path.getsize(ks_output))

Loading https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ ...
[SCRAPE FAILED] Kansas https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator(".fl-expandable-tree-accordion > .fl-accordion-item") to be visible

thread ok: False
file exists: True
file size: 0


In [9]:
# Detaillierte Navigation-Diagnose fuer Kansas
diag = {}

def _nav_diag_thread(url):
    async def _probe():
        from playwright.async_api import async_playwright
        async with async_playwright() as p:
            browser = await p.chromium.launch(
                headless=True,
                args=["--disable-blink-features=AutomationControlled", "--no-sandbox"],
            )
            context = await browser.new_context(
                viewport={"width": 1366, "height": 768},
                user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                locale="en-US",
            )
            page = await context.new_page()
            await page.add_init_script("""
                Object.defineProperty(navigator, 'webdriver', {get: () => false});
                window.navigator.chrome = { runtime: {} };
                Object.defineProperty(navigator, 'languages', {get: () => ['en-US', 'en']});
                Object.defineProperty(navigator, 'plugins', {get: () => [1,2,3,4,5]});
            """)
            resp = await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            diag["status"] = None if resp is None else resp.status
            diag["final_url"] = page.url
            diag["title"] = await page.title()
            diag["tree_root_count"] = await page.locator(".fl-expandable-tree-accordion").count()
            diag["tree_item_count"] = await page.locator(".fl-expandable-tree-accordion > .fl-accordion-item").count()
            diag["accordion_any_count"] = await page.locator(".fl-accordion-item").count()
            body = await page.locator("body").inner_text()
            diag["body_excerpt"] = body[:800]
            await context.close()
            await browser.close()

    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
    else:
        loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(_probe())
    finally:
        loop.close()

t = threading.Thread(target=_nav_diag_thread, args=(ks_url,))
t.start()
t.join()
diag

{'status': 403,
 'final_url': 'https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/?__cf_chl_rt_tk=Jlp14w8G7O2cVQ1qQ74Ag64ofbKrODjW09dtVINPdGI-1779435891-1.0.1.1-YEaJlMtPU.s19w5m.Hn4COe8HV_TjS74v0Map0nFnM4',
 'title': 'Just a moment...',
 'tree_root_count': 0,
 'tree_item_count': 0,
 'accordion_any_count': 0,
 'body_excerpt': 'codes.findlaw.com\nPerforming security verification\n\nThis website uses a security service to protect against malicious bots. This page is displayed while the website verifies you are not a bot.\n\nRay ID: 9ffa32b19dedd2a2\nPerformance and Security by Cloudflare\nPrivacy'}

In [10]:
# Test: Cloudflare-Challenge mit cloudscraper umgehen
import cloudscraper
scraper = cloudscraper.create_scraper(browser={"browser": "chrome", "platform": "windows", "mobile": False})
r = scraper.get(ks_url, timeout=60)
print("status:", r.status_code)
print("final url:", r.url)
print("title marker present:", "constitution" in r.text.lower())
print(r.text[:300])

status: 403
final url: https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/
title marker present: True
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal


In [11]:
# Test: Headful-Playwright fuer CF-Challenge (einmalig)
diag_headful = {}

def _nav_diag_headful(url):
    async def _probe():
        from playwright.async_api import async_playwright
        async with async_playwright() as p:
            browser = await p.chromium.launch(
                headless=False,
                args=["--disable-blink-features=AutomationControlled", "--no-sandbox"],
            )
            context = await browser.new_context(
                viewport={"width": 1366, "height": 900},
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
                locale="en-US",
            )
            page = await context.new_page()
            resp = await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_timeout(12000)
            diag_headful["status"] = None if resp is None else resp.status
            diag_headful["final_url"] = page.url
            diag_headful["title"] = await page.title()
            diag_headful["tree_item_count"] = await page.locator(".fl-expandable-tree-accordion > .fl-accordion-item").count()
            await context.close()
            await browser.close()

    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
    else:
        loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(_probe())
    finally:
        loop.close()

t = threading.Thread(target=_nav_diag_headful, args=(ks_url,))
t.start()
t.join()
diag_headful

{'status': 403,
 'final_url': 'https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/',
 'title': 'Just a moment...',
 'tree_item_count': 0}

In [12]:
# Test: Persistentes Browser-Profil (beste Chance gegen CF-Block)
diag_profile = {}
profile_dir = "findlaw_codes/playwright_profile"

def _nav_diag_profile(url):
    async def _probe():
        from playwright.async_api import async_playwright
        async with async_playwright() as p:
            context = await p.chromium.launch_persistent_context(
                user_data_dir=profile_dir,
                headless=False,
                viewport={"width": 1366, "height": 900},
                args=["--disable-blink-features=AutomationControlled", "--no-sandbox"],
            )
            page = context.pages[0] if context.pages else await context.new_page()
            resp = await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_timeout(12000)
            diag_profile["status"] = None if resp is None else resp.status
            diag_profile["final_url"] = page.url
            diag_profile["title"] = await page.title()
            diag_profile["tree_item_count"] = await page.locator(".fl-expandable-tree-accordion > .fl-accordion-item").count()
            await context.close()

    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
    else:
        loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(_probe())
    finally:
        loop.close()

t = threading.Thread(target=_nav_diag_profile, args=(ks_url,))
t.start()
t.join()
diag_profile

{'status': 403,
 'final_url': 'https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/',
 'title': 'Constitution of the State of Kansas | FindLaw',
 'tree_item_count': 19}

In [17]:
# One-off: Kansas Constitution mit persistentem Profil scrapen
from queue import Queue
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import time

def scrape_kansas_constitution_with_profile(output_file="us_codes/Kansas/cn.single.jsonl"):
    ks_url_local = "https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/"
    os.makedirs("us_codes/Kansas", exist_ok=True)
    if os.path.exists(output_file):
        os.remove(output_file)

    async def _collect_work():
        from playwright.async_api import async_playwright
        async with async_playwright() as p:
            context = await p.chromium.launch_persistent_context(
                user_data_dir="findlaw_codes/playwright_profile",
                headless=False,
                viewport={"width": 1366, "height": 900},
                args=["--disable-blink-features=AutomationControlled", "--no-sandbox"],
            )
            page = context.pages[0] if context.pages else await context.new_page()
            await page.goto(ks_url_local, wait_until="domcontentloaded", timeout=60000)

            # Cloudflare challenge may resolve after a short delay in persistent context
            ok = False
            for _ in range(12):
                if await page.locator(".fl-expandable-tree-accordion > .fl-accordion-item").count() > 0:
                    ok = True
                    break
                await page.wait_for_timeout(5000)

            if not ok:
                raise RuntimeError(f"Kansas tree not visible. title={await page.title()} url={page.url}")

            work_local = await ms_fl_scraper._collect_links_async(page, ks_url_local, ["KANSAS"], [1])
            await context.close()
            return work_local

    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
    else:
        loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        work = loop.run_until_complete(_collect_work())
    finally:
        loop.close()

    print("Collected links:", len(work))
    if not work:
        raise RuntimeError("No links collected for Kansas constitution")

    tmp_file = output_file + f".{int(time.time())}.tmp"
    if os.path.exists(tmp_file):
        os.remove(tmp_file)

    record_q = Queue()
    def _writer():
        with open(tmp_file, "w", encoding="utf-8") as f:
            while True:
                item = record_q.get()
                if item is None:
                    break
                f.write(item)

    wt = threading.Thread(target=_writer, daemon=True)
    wt.start()

    sess = requests.Session()
    sess.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    })

    pool = ThreadPoolExecutor(max_workers=8)
    futures = [
        pool.submit(ms_fl_scraper.fetch_leaf_threadsafe, sec_name, url, path, lex, "Kansas", sess, None, record_q)
        for sec_name, url, path, lex in work
    ]
    for _ in tqdm(as_completed(futures), total=len(futures), desc="Kansas leaf fetch"):
        pass
    pool.shutdown(wait=True)
    sess.close()

    record_q.put(None)
    wt.join()

    if os.path.exists(tmp_file) and os.path.getsize(tmp_file) > 0:
        os.replace(tmp_file, output_file)
        print("Saved:", output_file, "size:", os.path.getsize(output_file))
    else:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)
        raise RuntimeError("No output written")

thread_result = {}
def _run_one_off():
    try:
        scrape_kansas_constitution_with_profile()
        thread_result["ok"] = True
    except Exception as e:
        thread_result["ok"] = False
        thread_result["error"] = str(e)

t = threading.Thread(target=_run_one_off)
t.start()
t.join()
thread_result

Collected links: 158


Kansas leaf fetch:   0%|          | 0/158 [00:00<?, ?it/s]

{'ok': False, 'error': 'No output written'}

In [18]:
# Diagnose: erste Kansas-Leaf-URLs und HTTP-Antworten pruefen
leaf_probe = {}

def _collect_ks_work_for_probe():
    async def _collect():
        from playwright.async_api import async_playwright
        async with async_playwright() as p:
            context = await p.chromium.launch_persistent_context(
                user_data_dir="findlaw_codes/playwright_profile",
                headless=False,
                viewport={"width": 1366, "height": 900},
                args=["--disable-blink-features=AutomationControlled", "--no-sandbox"],
            )
            page = context.pages[0] if context.pages else await context.new_page()
            await page.goto(ks_url, wait_until="domcontentloaded", timeout=60000)
            for _ in range(10):
                if await page.locator(".fl-expandable-tree-accordion > .fl-accordion-item").count() > 0:
                    break
                await page.wait_for_timeout(5000)
            w = await ms_fl_scraper._collect_links_async(page, ks_url, ["KANSAS"], [1])
            await context.close()
            return w

    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
    else:
        loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(_collect())
    finally:
        loop.close()

box = {}
def _runner_probe():
    box["work"] = _collect_ks_work_for_probe()

tt = threading.Thread(target=_runner_probe)
tt.start()
tt.join()
work_probe = box.get("work", [])
print("work count:", len(work_probe))
print("first 3 urls:")
for x in work_probe[:3]:
    print(x[1])

s = requests.Session()
s.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"})
probe_rows = []
for x in work_probe[:3]:
    u = x[1]
    rr = s.get(u, timeout=30)
    probe_rows.append((u, rr.status_code, rr.url, rr.text[:120]))
s.close()
probe_rows

work count: 158
first 3 urls:
https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ks-const-ordinance/
https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ks-const-bill-of-rights-sect-1/
https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ks-const-bill-of-rights-sect-2/


[('https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ks-const-ordinance/',
  403,
  'https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ks-const-ordinance/',
  '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/htm'),
 ('https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ks-const-bill-of-rights-sect-1/',
  403,
  'https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ks-const-bill-of-rights-sect-1/',
  '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/htm'),
 ('https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ks-const-bill-of-rights-sect-2/',
  403,
  'https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ks-const-bill-of-rights-sect-2/',
  '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/htm')]

In [19]:
# Workaround: Kansas komplett ueber Playwright (ohne requests fuer Leafs)
import json
from bs4 import BeautifulSoup

def scrape_kansas_constitution_playwright_only(output_file="us_codes/Kansas/cn.playwright.jsonl"):
    os.makedirs("us_codes/Kansas", exist_ok=True)

    async def _run():
        from playwright.async_api import async_playwright
        async with async_playwright() as p:
            context = await p.chromium.launch_persistent_context(
                user_data_dir="findlaw_codes/playwright_profile",
                headless=False,
                viewport={"width": 1366, "height": 900},
                args=["--disable-blink-features=AutomationControlled", "--no-sandbox"],
            )
            page = context.pages[0] if context.pages else await context.new_page()

            root_url = "https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/"
            await page.goto(root_url, wait_until="domcontentloaded", timeout=60000)
            ok = False
            for _ in range(12):
                if await page.locator(".fl-expandable-tree-accordion > .fl-accordion-item").count() > 0:
                    ok = True
                    break
                await page.wait_for_timeout(5000)
            if not ok:
                raise RuntimeError(f"Kansas tree not visible. title={await page.title()} url={page.url}")

            work = await ms_fl_scraper._collect_links_async(page, root_url, ["KANSAS"], [1])
            print("Collected links:", len(work))
            if not work:
                raise RuntimeError("No links collected for Kansas")

            tmp_file = output_file + f".{int(time.time())}.tmp"
            with open(tmp_file, "w", encoding="utf-8") as f_out:
                for sec_name, sec_url, path_so_far, lex_path in tqdm(work, desc="Kansas playwright leaf fetch"):
                    await page.goto(sec_url, wait_until="domcontentloaded", timeout=60000)
                    html = await page.content()
                    soup = BeautifulSoup(html, "html.parser")

                    h1 = soup.select_one("h1")
                    if h1 is None:
                        continue
                    statute_name = h1.get_text(strip=True)
                    content_div = ms_fl_scraper.clean_paragraphs(soup)
                    if not content_div:
                        continue

                    parsed_heading = ms_fl_scraper._parse_statute_heading(statute_name, sec_name)
                    hierarchy = ms_fl_scraper._build_hierarchy_metadata(
                        path_so_far,
                        lex_path,
                        "Kansas",
                        leaf_title=parsed_heading.get("article_title", ""),
                        leaf_heading=parsed_heading.get("heading", ""),
                        leaf_code=parsed_heading.get("article_code", "") or sec_name,
                    )

                    data = {
                        "url": sec_url,
                        "state": "KANSAS",
                        "path": "›".join(path_so_far),
                        "path_nodes": path_so_far,
                        "parent_path": "›".join(path_so_far[:-1]) if len(path_so_far) > 1 else "",
                        "title": f"KANSAS Statutes › {' › '.join(path_so_far)}",
                        "univ_cite": False,
                        "citation": f"KANSAS Stat § {sec_name} (2023)",
                        "statute_name": statute_name,
                        "article_heading": parsed_heading.get("heading", ""),
                        "article_code": parsed_heading.get("article_code", "") or sec_name,
                        "article_title": parsed_heading.get("article_title", ""),
                        "content": content_div,
                        "lex_path": lex_path,
                        "hierarchy": hierarchy,
                        "parent_nodes": hierarchy["parent_nodes"],
                    }
                    f_out.write(json.dumps(data, ensure_ascii=False) + "\n")

            await context.close()

            if os.path.exists(output_file):
                os.remove(output_file)
            os.replace(tmp_file, output_file)
            return output_file

    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
    else:
        loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(_run())
    finally:
        loop.close()

result_one = {}
def _run_playwright_only():
    try:
        out = scrape_kansas_constitution_playwright_only()
        result_one["ok"] = True
        result_one["output"] = out
        result_one["size"] = os.path.getsize(out) if os.path.exists(out) else None
    except Exception as e:
        result_one["ok"] = False
        result_one["error"] = str(e)

t = threading.Thread(target=_run_playwright_only)
t.start()
t.join()
result_one

Collected links: 158


Kansas playwright leaf fetch:   0%|          | 0/158 [00:00<?, ?it/s]

{'ok': True, 'output': 'us_codes/Kansas/cn.playwright.jsonl', 'size': 390369}

In [33]:
# Notebook-Validation des aktuellen Scrapers auf Kansas
import traceback
nb_result = {}
def _nb_run():
    try:
        target = 'us_codes/Kansas/cn.loopfixed.jsonl'
        if os.path.exists(target):
            os.remove(target)
        nb_result['ok'] = run_scraper(ks_url, ks_state, target)
    except Exception as e:
        nb_result['ok'] = False
        nb_result['error'] = str(e)
        nb_result['traceback'] = traceback.format_exc()

t = threading.Thread(target=_nb_run)
t.start()
t.join()
nb_result

Loading https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ ...
Found 158 statutes. Fetching...


Fetching (Playwright): 100%|██████████| 158/158 [05:33<00:00,  2.11s/it]


Done. Saved to us_codes/Kansas/cn.loopfixed.jsonl


{'ok': True}

In [28]:
# Direkte Traceback-Diagnose ohne run_scraper-Wrapper
import traceback
direct_result = {}
def _direct_run():
    try:
        asyncio.run(ms_fl_scraper.scrape_section_url_async(ks_url, ks_state, 'us_codes/Kansas/cn.nbtest2.jsonl'))
        direct_result['ok'] = True
    except Exception as e:
        direct_result['ok'] = False
        direct_result['error'] = str(e)
        direct_result['traceback'] = traceback.format_exc()

t = threading.Thread(target=_direct_run)
t.start()
t.join()
direct_result

Task exception was never retrieved
future: <Task finished name='Task-656' coro=<Connection.run() done, defined at d:\Sync\Uni\Promotion\research\lawscraper\.venv\Lib\site-packages\playwright\_impl\_connection.py:305> exception=NotImplementedError()>
Traceback (most recent call last):
  File "d:\Sync\Uni\Promotion\research\lawscraper\.venv\Lib\site-packages\playwright\_impl\_connection.py", line 312, in run
    await self._transport.connect()
  File "d:\Sync\Uni\Promotion\research\lawscraper\.venv\Lib\site-packages\playwright\_impl\_transport.py", line 133, in connect
    raise exc
  File "d:\Sync\Uni\Promotion\research\lawscraper\.venv\Lib\site-packages\playwright\_impl\_transport.py", line 120, in connect
    self._proc = await asyncio.create_subprocess_exec(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "C:\Users\TimoS\AppData\Local\Programs\Python\Python313\Lib\asyncio\subprocess.py", line 224, in create_subprocess_exec
    transport, 

{'ok': False,
 'error': '',
 'traceback': 'Traceback (most recent call last):\n  File "C:\\Users\\TimoS\\AppData\\Local\\Temp\\ipykernel_25184\\1786357956.py", line 6, in _direct_run\n    asyncio.run(ms_fl_scraper.scrape_section_url_async(ks_url, ks_state, \'us_codes/Kansas/cn.nbtest2.jsonl\'))\n    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "C:\\Users\\TimoS\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\asyncio\\runners.py", line 195, in run\n    return runner.run(main)\n           ~~~~~~~~~~^^^^^^\n  File "C:\\Users\\TimoS\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\asyncio\\runners.py", line 118, in run\n    return self._loop.run_until_complete(task)\n           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^\n  File "C:\\Users\\TimoS\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\asyncio\\base_events.py", line 725, in run_until_complete\n    return future.result()\n           ~~~~~~~~~~~~~^^\n  File "d:\\Syn

In [8]:
# Quick check: Kansas mit neuem Fallback nach state_codes
import importlib
import os
import threading
import ms_fl_scraper
ms_fl_scraper = importlib.reload(ms_fl_scraper)
scrape_section_url_async = ms_fl_scraper.scrape_section_url_async

ks_url_local = "https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/"
ks_state_local = "Kansas"
target = "state_codes/Kansas/cn.loopfixed.jsonl"
os.makedirs("state_codes/Kansas", exist_ok=True)
if os.path.exists(target):
    os.remove(target)

box = {}
def _run():
    box["ok"] = run_scraper(ks_url_local, ks_state_local, target)

t = threading.Thread(target=_run)
t.start()
t.join()
print("ok:", box.get("ok"))
print("exists:", os.path.exists(target))
if os.path.exists(target):
    print("size:", os.path.getsize(target))

Loading https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ ...
Found 158 statutes. Fetching...


Fetching (Playwright): 100%|██████████| 158/158 [32:17<00:00, 12.26s/it]


[SCRAPE FAILED] Kansas https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/: Playwright fallback produced no records for https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/
ok: False
exists: False


In [9]:
# Quick leaf diagnosis in same persistent context
diag_leaf = {}

def _diag_leaf_thread():
    async def _run():
        from playwright.async_api import async_playwright
        async with async_playwright() as p:
            context = await p.chromium.launch_persistent_context(
                user_data_dir="findlaw_codes/playwright_profile",
                headless=False,
                args=["--disable-blink-features=AutomationControlled", "--no-sandbox"],
                viewport={"width": 1366, "height": 900},
            )
            page = context.pages[0] if context.pages else await context.new_page()
            root = "https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/"
            await page.goto(root, wait_until="domcontentloaded", timeout=60000)
            for _ in range(12):
                if await page.locator(".fl-expandable-tree-accordion > .fl-accordion-item").count() > 0:
                    break
                await page.wait_for_timeout(5000)
            work = await ms_fl_scraper._collect_links_async(page, root, ["KANSAS"], [1])
            first_url = work[0][1] if work else None
            diag_leaf["work_count"] = len(work)
            diag_leaf["first_url"] = first_url
            if first_url:
                await page.goto(first_url, wait_until="domcontentloaded", timeout=60000)
                await page.wait_for_timeout(5000)
                html = await page.content()
                diag_leaf["title"] = await page.title()
                diag_leaf["blocked"] = ms_fl_scraper._is_blocked_findlaw_response(html)
                diag_leaf["has_h1"] = (await page.locator("h1").count()) > 0
                diag_leaf["has_codes_content_p"] = (await page.locator("div.codes-content p").count()) > 0
                diag_leaf["html_excerpt"] = html[:400]
            await context.close()

    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
    else:
        loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(_run())
    finally:
        loop.close()

t = threading.Thread(target=_diag_leaf_thread)
t.start()
t.join()
diag_leaf

{'work_count': 158,
 'first_url': 'https://codes.findlaw.com/ks/constitution-of-the-state-of-kansas/ks-const-ordinance/',
 'title': 'Constitution of the State of Kansas Ordinance | FindLaw',
 'blocked': True,
 'has_h1': True,
 'has_codes_content_p': True,
 'html_excerpt': '<!DOCTYPE html><html lang="en-US" style="--screen-height: 900px;"><head class="at-element-marker">\n    <meta charset="UTF-8">\n\n    <meta http-equiv="X-UA-Compatible" content="ie=edge">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n\n    <meta name="csrf-token" content="d8f001e4e0">\n\n\t<link rel="search" type="application/opensearchdescription+xml" title="LATL Search" hre'}

In [10]:
# Re-check blocker classifier on first Kansas leaf
import importlib
import ms_fl_scraper
ms_fl_scraper = importlib.reload(ms_fl_scraper)
html_sample = diag_leaf.get("html_excerpt", "")
print("blocked_on_excerpt:", ms_fl_scraper._is_blocked_findlaw_response(html_sample))

blocked_on_excerpt: False
